In [1]:
# ! pip install flax tqdm

In [2]:
import os
import tempfile

import flax.linen as nn
import jax
import jax.numpy as jnp
import numpy as np
import optax
from tqdm.notebook import tqdm

from jax_async_ckpt.plugin import JaxAsyncCheckpointer

In [3]:
class SyntheticMLP(nn.Module):
    features: int
    num_classes: int

    @nn.compact
    def __call__(self, x):
        x = nn.Dense(features=self.features)(x)
        x = nn.relu(x)
        x = nn.Dense(features=self.features)(x)
        x = nn.relu(x)
        x = nn.Dense(features=self.num_classes)(x)
        return x


@jax.jit
def compute_grads(params, x, y):
    def loss_fn(p):
        logits = SyntheticMLP(
            features=params["Dense_0"]["kernel"].shape[1], num_classes=10
        ).apply({"params": p}, x)
        return jnp.mean(
            optax.softmax_cross_entropy_with_integer_labels(
                logits=logits,
                labels=y,
            )
        )

    grads = jax.grad(loss_fn)(params)
    return grads

In [4]:
# Configuration
hidden_dim = 4096
num_steps = 20
ckpt_interval = 2
max_buffer_mb = 1024

# Initialize Engine
checkpointer = JaxAsyncCheckpointer(max_buffer_bytes=max_buffer_mb * 1024 * 1024)

# Initialize Model & Allocate Parameter PyTree
model = SyntheticMLP(features=hidden_dim, num_classes=10)
key = jax.random.PRNGKey(42)
dummy_x = jnp.ones((128, hidden_dim), dtype=jnp.float32)
dummy_y = jnp.zeros((128,), dtype=jnp.int32)

variables = model.init(key, dummy_x)
params = variables["params"]

# Initialize Optimizer
tx = optax.adam(learning_rate=1e-3)
opt_state = tx.init(params)

## Synchronous Checkpointing

In [5]:
tmpdir_sync_ctx = tempfile.TemporaryDirectory()
sync_target_dir = tmpdir_sync_ctx.name
sync_base_path = os.path.join(sync_target_dir, "flax_sync_checkpoint")

pbar_sync = tqdm(range(num_steps), desc="Synchronous Training Steps")

for step in pbar_sync:
    # 1. Compute Gradients
    grads = compute_grads(params, dummy_x, dummy_y)

    # 2. Apply Optimizer Updates
    updates, opt_state = tx.update(grads, opt_state, params)
    params = optax.apply_updates(params, updates)

    # 3. Save PyTree Checkpoint Synchronously (Host-Blocking)
    if step % ckpt_interval == 0:
        # Force host-side block & copy back to CPU memory
        params_cpu = jax.tree.map(
            lambda leaf: np.array(jax.block_until_ready(leaf)), params
        )

        # Write to disk on host thread
        np.savez(f"{sync_base_path}_step_{step}.npz", **params_cpu)

tmpdir_sync_ctx.cleanup()
print("Synchronous training and checkpointing complete successfully!")

Synchronous Training Steps:   0%|          | 0/20 [00:00<?, ?it/s]

Synchronous training and checkpointing complete successfully!


## Asynchronous Checkpointing

In [6]:
tmpdir_async_ctx = tempfile.TemporaryDirectory()
target_dir = tmpdir_async_ctx.name
base_path = os.path.join(target_dir, "flax_async_checkpoint")

pbar_async = tqdm(range(num_steps), desc="Asynchronous Training Steps")

for step in pbar_async:
    # 1. Compute Gradients
    grads = compute_grads(params, dummy_x, dummy_y)

    # 2. Apply Optimizer Updates
    updates, opt_state = tx.update(grads, opt_state, params)
    params = optax.apply_updates(params, updates)

    # 3. Save PyTree Checkpoint Asynchronously
    if step % ckpt_interval == 0:
        checkpointer.save_pytree_async(params, f"{base_path}_step_{step}")

# Flush background C++ io_uring write queue before exit
checkpointer.wait_all()

tmpdir_async_ctx.cleanup()
print("Asynchronous training and checkpointing complete successfully!")

Asynchronous Training Steps:   0%|          | 0/20 [00:00<?, ?it/s]

Asynchronous training and checkpointing complete successfully!
